In [20]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error
import matplotlib.pyplot as plt

Connect SQLite3 to pandas to get data.

In [2]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../../market_data.db')

In [3]:
sales_df = pd.read_sql("SELECT * FROM sales_history", conn)
competitor_df = pd.read_sql("SELECT * FROM competitor_intelligence", conn)
inventory_df = pd.read_sql("SELECT * FROM inventory_cost", conn)

In [4]:
sales_df.head()
competitor_df.head()
inventory_df.head()



,product_id,current_stock,base_cost
0,WIDGET-A,100,10.0
1,GADGET-B,50,25.0
2,SENSOR-C,200,5.0


Merge all three tables and Start of Day-5


In [5]:
df=sales_df.merge(
    competitor_df,
    on='product_id',
    how='left'
).merge(
    inventory_df,
    on='product_id',
    how='left'
)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0


Sort the values by product ID and Date.

In [6]:
df['date']=pd.to_datetime(df['date'])
df=df.sort_values(by=['product_id','date'])
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0


New Column Price Gap and Price Gap Percentage.


In [7]:
df['price_gap']=df['price']-df['competitor_price']
df['price_gap_percentage']=round((df['price']-df['competitor_price'])/df['competitor_price']*100,2)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0,5.0,14.29
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0,3.0,8.57
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,-3.5,-18.92
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,1.5,8.11


New Column : Inventory Velocity

In [8]:
# 1. Calculate a 7-day rolling average of units sold per product
df['rolling_units_7d'] = df.groupby('product_id')['units_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# 2. Calculate Velocity: How much of our current stock do we sell daily (on average)?
# We use a small epsilon (0.0001) or fillna to avoid division by zero errors
df['inventory_velocity'] = df['rolling_units_7d'] / df['current_stock'].replace(0, 1)

# 3. Round for cleanliness
df['inventory_velocity'] = df['inventory_velocity'].round(3)

df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage,rolling_units_7d,inventory_velocity
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0,5.0,14.29,20.0,0.40
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0,3.0,8.57,21.0,0.42
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,-3.5,-18.92,50.0,0.50
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,1.5,8.11,40.0,0.40


New Column: Price Elasticity


In [9]:
# 1. Calculate percentage change in quantity and price
df['pct_change_q'] = df.groupby('product_id')['units_sold'].pct_change()
df['pct_change_p'] = df.groupby('product_id')['price'].pct_change()

# 2. Elasticity = (% Change in Q) / (% Change in P)
df['price_elasticity'] = df['pct_change_q'] / df['pct_change_p']

# 3. Clean up: replace infinite values (from 0 price change) with 0
import numpy as np
df['price_elasticity'] = df['price_elasticity'].replace([np.inf, -np.inf], 0).fillna(0)

df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage,rolling_units_7d,inventory_velocity,pct_change_q,pct_change_p,price_elasticity
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0,5.0,14.29,20.0,0.40,NaN,NaN,0.0
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0,3.0,8.57,21.0,0.42,0.1,-0.050000,-2.0
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,-3.5,-18.92,50.0,0.50,NaN,NaN,0.0
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,1.5,8.11,40.0,0.40,-0.4,0.333333,-1.2


ML: Linear Regression 
Some Important Answers:
If a simple linear regression performs worse than random, it indicates a fundamental issue in the pipeline — either the data quality is poor, the features do not encode the underlying demand logic, or the target variable is incorrectly constructed (often due to leakage or misalignment).
In such a case, adding a more complex model would only amplify noise, not improve performance.
A random train-test split is dangerous because this is time-series data. Randomly mixing past and future observations causes data leakage, allowing the model to indirectly see future demand patterns during training. A time-based split preserves causality by training on earlier dates and validating on later dates.


Column Name: Demand Trend

In [10]:
# Calculate the % change in the 7-day rolling average
# This tells us if the 'pace' of sales is accelerating or slowing down
df['demand_trend'] = df.groupby('product_id')['rolling_units_7d'].pct_change()

# Clean up: Replace NaN (from the first row) and any division-by-zero errors with 0
df['demand_trend'] = df['demand_trend'].fillna(0).replace([float('inf'), float('-inf')], 0)

print("Column 'demand_trend' successfully added.")

Column 'demand_trend' successfully added.


ML: Features and Prediction

In [18]:
df["units_sold_next_7_days"] = (
    df.groupby("product_id")["units_sold"]
      .rolling(window=7, min_periods=7)
      .sum()
      .shift(-7)
      .reset_index(level=0, drop=True)
)

In [19]:
feature_cols=[
    'price',
    'price_elasticity',
    'inventory_velocity',
    'price_gap',
    'demand_trend'
]
df_clean=df.dropna(subset=feature_cols + ['units_sold_next_7_days'])
X = df_clean[feature_cols]
y = df_clean['units_sold_next_7_days']
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns in X:", X.columns.tolist())

X shape: (0, 5)
y shape: (0,)
Columns in X: ['price', 'price_elasticity', 'inventory_velocity', 'price_gap', 'demand_trend']


Phase-2 Day-6: Baseline Model Sanity Check